In [0]:
# List all files in our volume
display(
    dbutils.fs.ls("/Volumes/workspace/default/retail_data/")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/retail_data/customers.csv,customers.csv,4149,1791657338000
dbfs:/Volumes/workspace/default/retail_data/results (1).csv,results (1).csv,160,1791657338000
dbfs:/Volumes/workspace/default/retail_data/results (2).csv,results (2).csv,786,1791657339000
dbfs:/Volumes/workspace/default/retail_data/results.csv,results.csv,160,1791657338000


In [0]:
# Load the customers dataset
customers = spark.read.csv(
    "/Volumes/workspace/default/retail_data/customers.csv",
    header=True,
    inferSchema=True
)

display(customers)

# Check column names and data types
customers.printSchema()

# Count total rows
print("Total customers:", customers.count())

customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)

Total customers: 50


In [0]:

# Base path for uploaded CSV files
base_path = "/Volumes/workspace/default/retail_data/"

# Read raw CSV files
df_customers_bronze = spark.read.csv(
    base_path + "customers.csv",
    header=True,
    inferSchema=True
)

df_products_bronze = spark.read.csv(
    base_path + "products.csv",
    header=True,
    inferSchema=True
)

df_stores_bronze = spark.read.csv(
    base_path + "stores.csv",
    header=True,
    inferSchema=True
)

df_transactions_bronze = spark.read.csv(
    base_path + "transactions.csv",
    header=True,
    inferSchema=True
)

# Preview the datasets
display(df_customers_bronze)
display(df_products_bronze)
display(df_stores_bronze)
display(df_transactions_bronze)


customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.0
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.0
6,Water Bottle,Fitness,299.0
7,Smartwatch,Electronics,4999.0
8,Desk Organizer,Accessories,399.0
9,Dumbbell Set,Fitness,1999.0
10,Pen Drive 32GB,Electronics,599.0


store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


In [0]:
# Fix DELTA_METADATA_MISMATCH for bronze_products by dropping the table first
spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_products")

df_customers_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_customers"
)

df_products_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_products"
)

df_stores_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_stores"
)

df_transactions_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_transactions"
)

print("Bronze layer created successfully!")


Bronze layer created successfully!


In [0]:
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_products,false
default,bronze_stores,false
default,bronze_transactions,false


In [0]:
display(spark.sql("SELECT * FROM workspace.default.bronze_customers LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_products LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_stores LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_transactions LIMIT 10"))

customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.0
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.0
6,Water Bottle,Fitness,299.0
7,Smartwatch,Electronics,4999.0
8,Desk Organizer,Accessories,399.0
9,Dumbbell Set,Fitness,1999.0
10,Pen Drive 32GB,Electronics,599.0


store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


In [0]:
display(spark.sql("SELECT * FROM workspace.default.bronze_customers LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_products LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_stores LIMIT 10"))

display(spark.sql("SELECT * FROM workspace.default.bronze_transactions LIMIT 10"))

customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28
106,Ajay,Mishra,user106@example.com,9832109876,Pune,2024-03-10
107,Priya,Kapoor,user107@example.com,9821098765,Ahmedabad,2023-05-12
108,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19
109,Pooja,Mehta,user109@example.com,9809876543,Delhi,2024-04-01
110,Deepak,Nair,user110@example.com,9798765432,Mumbai,2023-10-14


product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.0
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.0
6,Water Bottle,Fitness,299.0
7,Smartwatch,Electronics,4999.0
8,Desk Organizer,Accessories,399.0
9,Dumbbell Set,Fitness,1999.0
10,Pen Drive 32GB,Electronics,599.0


store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17
6,110,7,3,5,2025-01-04
7,110,7,2,5,2025-01-01
8,126,7,5,2,2025-06-08
9,123,1,3,2,2024-10-08
10,124,2,2,5,2024-08-27


In [0]:

# Inspect the actual schema of all four Bronze tables

table_names = [
    "bronze_customers",
    "bronze_products",
    "bronze_stores",
    "bronze_transactions"
]

for name in table_names:
    print(f"\n{'=' * 40}")
    print("TABLE:", name)

    df = spark.table(f"workspace.default.{name}")

    print("Columns:", df.columns)
    df.printSchema()
    display(df.limit(5))



TABLE: bronze_customers
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'registration_date']
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)



customer_id,first_name,last_name,email,phone,city,registration_date
101,Ravi,Yadav,user101@example.com,9887654321,Delhi,2023-09-14
102,Nina,Joshi,user102@example.com,9876543210,Mumbai,2024-01-21
103,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10
104,Karan,Patel,user104@example.com,9854321098,Hyderabad,2024-02-05
105,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28



TABLE: bronze_products
Columns: ['product_id', 'product_name', 'category', 'price']
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



product_id,product_name,category,price
1,Wireless Mouse,Electronics,799.99
2,Bluetooth Speaker,Electronics,1299.49
3,Yoga Mat,Fitness,499.0
4,Laptop Stand,Accessories,999.99
5,Notebook Set,Stationery,149.0



TABLE: bronze_stores
Columns: ['store_id', 'store_name', 'location']
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- location: string (nullable = true)



store_id,store_name,location
1,City Mall Store,Mumbai
2,High Street Store,Delhi
3,Tech World Outlet,Bangalore
4,Downtown Mini Store,Pune
5,Mega Plaza,Chennai



TABLE: bronze_transactions
Columns: ['transaction_id', 'customer_id', 'product_id', 'store_id', 'quantity', 'transaction_date']
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,127,8,4,4,2025-03-31
2,105,3,4,5,2024-11-12
3,116,2,2,3,2025-05-01
4,120,8,1,1,2024-11-02
5,105,5,2,1,2025-03-17


In [0]:

from pyspark.sql.functions import col

# 1. Read Bronze Delta tables
df_transactions = spark.table(
    "workspace.default.bronze_transactions"
)

df_products = spark.table(
    "workspace.default.bronze_products"
)

df_stores = spark.table(
    "workspace.default.bronze_stores"
)

df_customers = spark.table(
    "workspace.default.bronze_customers"
)

# 2. Clean Transactions
df_transactions = (
    df_transactions
    .select(
        col("transaction_id").cast("int"),
        col("customer_id").cast("int"),
        col("product_id").cast("int"),
        col("store_id").cast("int"),
        col("quantity").cast("int"),
        col("transaction_date").cast("date")
    )
    .dropDuplicates(["transaction_id"])
)

# 3. Clean Products
df_products = (
    df_products
    .select(
        col("product_id").cast("int"),
        col("product_name"),
        col("category"),
        col("price").cast("double")
    )
    .dropDuplicates(["product_id"])
)

# 4. Clean Stores
df_stores = (
    df_stores
    .select(
        col("store_id").cast("int"),
        col("store_name"),
        col("location")
    )
    .dropDuplicates(["store_id"])
)

# 5. Clean Customers
df_customers = (
    df_customers
    .select(
        col("customer_id").cast("int"),
        col("first_name"),
        col("last_name"),
        col("email"),
        col("phone"),
        col("city"),
        col("registration_date").cast("date")
    )
    .dropDuplicates(["customer_id"])
)

# 6. Join all datasets
df_silver = (
    df_transactions
    .join(df_customers, "customer_id", "left")
    .join(df_products, "product_id", "left")
    .join(df_stores, "store_id", "left")
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
)

# 7. Preview Silver data
display(df_silver)

print("Silver row count:", df_silver.count())


store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,phone,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,108,12,4,2025-05-26,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
2,6,118,16,4,2024-11-29,Vikram,Mehta,user118@example.com,9710987654,Mumbai,2023-04-22,Water Bottle,Fitness,299.0,High Street Store,Delhi,1196.0
2,5,105,5,1,2025-03-17,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,8,127,1,4,2025-03-31,Rohit,Mehta,user127@example.com,9621098765,Bangalore,2023-08-30,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1596.0
2,2,116,3,3,2025-05-01,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,3898.4700000000003
1,8,127,27,1,2024-10-10,Rohit,Mehta,user127@example.com,9621098765,Bangalore,2023-08-30,Desk Organizer,Accessories,399.0,City Mall Store,Mumbai,399.0
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,9598765432,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,9709876543,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0


Silver row count: 30


In [0]:

# Save Silver as a managed Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.retail_silver_cleaned")

# Verify the saved table
display(
    spark.sql("""
        SELECT *
        FROM workspace.default.retail_silver_cleaned
        LIMIT 10
    """)
)

print("Silver table created successfully!")


store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,phone,city,registration_date,product_name,category,price,store_name,location,total_amount
1,5,108,12,4,2025-05-26,Rahul,Verma,user108@example.com,9810987654,Kolkata,2023-08-19,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
4,1,103,18,3,2024-09-05,Sonal,Sharma,user103@example.com,9865432109,Bangalore,2023-07-10,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.9700000000003
2,6,118,16,4,2024-11-29,Vikram,Mehta,user118@example.com,9710987654,Mumbai,2023-04-22,Water Bottle,Fitness,299.0,High Street Store,Delhi,1196.0
2,5,105,5,1,2025-03-17,Riya,Singh,user105@example.com,9843210987,Chennai,2023-06-28,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
2,2,124,10,5,2024-08-27,Kavita,Sharma,user124@example.com,9654321098,Kolkata,2023-11-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,6497.45
4,8,127,1,4,2025-03-31,Rohit,Mehta,user127@example.com,9621098765,Bangalore,2023-08-30,Desk Organizer,Accessories,399.0,Downtown Mini Store,Pune,1596.0
2,2,116,3,3,2025-05-01,Rakesh,Kapoor,user116@example.com,9732109876,Kolkata,2023-06-15,Bluetooth Speaker,Electronics,1299.49,High Street Store,Delhi,3898.4700000000003
1,8,127,27,1,2024-10-10,Rohit,Mehta,user127@example.com,9621098765,Bangalore,2023-08-30,Desk Organizer,Accessories,399.0,City Mall Store,Mumbai,399.0
1,5,130,20,2,2024-07-30,Tanvi,Kapoor,user130@example.com,9598765432,Pune,2023-12-22,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,298.0
4,9,119,29,2,2025-06-03,Kunal,Nair,user119@example.com,9709876543,Bangalore,2023-08-10,Dumbbell Set,Fitness,1999.0,Downtown Mini Store,Pune,3998.0


Silver table created successfully!


In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(total_amount) AS total_sales,
        COUNT(DISTINCT transaction_id) AS unique_transactions
    FROM workspace.default.retail_silver_cleaned
"""))

total_rows,total_sales,unique_transactions
30,134570.74,30


In [0]:

from pyspark.sql.functions import sum, countDistinct, avg

# Read cleaned data from the Silver table
silver_df = spark.table(
    "workspace.default.retail_silver_cleaned"
)

# Aggregate sales for business reporting
gold_df = silver_df.groupBy(
    "transaction_date",
    "product_id",
    "product_name",
    "category",
    "store_id",
    "store_name",
    "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)

# Preview Gold data
display(gold_df)

print("Gold row count:", gold_df.count())


transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2025-03-17,5,Notebook Set,Stationery,2,High Street Store,Delhi,1,149.0,1,149.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2025-03-31,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,4,1596.0,1,1596.0
2025-05-01,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,3,3898.4700000000003,1,3898.4700000000003
2024-10-10,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-07-30,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,2,298.0,1,298.0
2025-06-03,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0


Gold row count: 30


In [0]:

# Save Gold as a managed Delta table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.retail_gold_sales_summary")

# Verify the table
display(
    spark.sql("""
        SELECT *
        FROM workspace.default.retail_gold_sales_summary
        LIMIT 20
    """)
)

print("Gold table created successfully!")


transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-11-12,3,Yoga Mat,Fitness,4,Downtown Mini Store,Pune,5,2495.0,1,2495.0
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98


Gold table created successfully!


In [0]:

display(spark.sql("""
    SELECT
        category,
        ROUND(SUM(total_sales_amount)) AS revenue
    FROM workspace.default.retail_gold_sales_summary
    GROUP BY category
    ORDER BY revenue DESC
"""))


category,revenue
Electronics,99781.0
Fitness,23772.0
Accessories,9975.0
Stationery,1043.0


In [0]:

display(spark.sql("""
    SELECT
        store_id,
        store_name,
        location,
        ROUND(SUM(total_sales_amount)) AS revenue
    FROM workspace.default.retail_gold_sales_summary
    GROUP BY store_id, store_name, location
    ORDER BY revenue DESC
"""))


store_id,store_name,location,revenue
3,Tech World Outlet,Bangalore,42187.0
2,High Street Store,Delhi,37334.0
5,Mega Plaza,Chennai,29285.0
4,Downtown Mini Store,Pune,24073.0
1,City Mall Store,Mumbai,1692.0


In [0]:

display(spark.sql("""
    SELECT
        transaction_date,
        SUM(total_sales_amount) AS daily_revenue,
        SUM(total_quantity_sold) AS units_sold
    FROM workspace.default.retail_gold_sales_summary
    GROUP BY transaction_date
    ORDER BY transaction_date
"""))


transaction_date,daily_revenue,units_sold
2024-07-10,1995.0,5
2024-07-14,2395.99,5
2024-07-17,3999.95,5
2024-07-30,298.0,2
2024-08-11,1599.98,2
2024-08-27,6497.45,5
2024-09-05,2399.9700000000003,3
2024-09-21,598.0,2
2024-10-02,3999.95,5
2024-10-08,1599.98,2


In [0]:
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_products,false
default,bronze_stores,false
default,bronze_transactions,false
default,retail_gold_sales_summary,false
default,retail_silver_cleaned,false


In [0]:
# Load the merged Gold table
df = spark.table("workspace.default.retail_gold_sales_summary")

# Convert the table to a single CSV file
df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/workspace/default/retail_data/retail_gold_csv")

print("CSV export completed!")

CSV export completed!


In [0]:

# Display the Gold table
df = spark.table("workspace.default.retail_gold_sales_summary")
display(df)


transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-11-12,3,Yoga Mat,Fitness,4,Downtown Mini Store,Pune,5,2495.0,1,2495.0
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
